<a href="https://colab.research.google.com/github/kongcheng65333/Hax_extend/blob/main/Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import subprocess
import os
import time
import threading
import requests

# ========== 配置区，替换你的HF Token ==========
HF_TOKEN = "hf_这里粘贴你自己的token"
COMFY_PATH = "/root/comfy/ComfyUI"
# Colab保活间隔（秒），默认60秒发送一次保活请求
KEEP_ALIVE_INTERVAL = 60
# wget下载重试次数
WGET_RETRY_COUNT = 5
# ===========================================

# 创建所有目标文件夹
os.makedirs(f"{COMFY_PATH}/models/checkpoints", exist_ok=True)
os.makedirs(f"{COMFY_PATH}/models/latent_upscale_models", exist_ok=True)
os.makedirs(f"{COMFY_PATH}/models/loras", exist_ok=True)

# Colab 后台保活函数（独立线程运行，不阻塞下载）
def colab_keep_alive():
    """持续发送请求防止Colab闲置断开内核"""
    url = "https://www.google.com"
    while True:
        try:
            requests.get(url, timeout=10)
            print(f"\n[保活心跳] {time.strftime('%Y-%m-%d %H:%M:%S')} 内核保活成功")
        except Exception as e:
            print(f"\n[保活心跳] 保活请求异常: {str(e)}，继续循环")
        time.sleep(KEEP_ALIVE_INTERVAL)

# 定义下载函数：带鉴权、断点续传、实时网速进度、重试
def download_file(url, save_path):
    print("\n" + "="*60)
    print(f"【开始任务】{url}")
    print(f"【保存路径】{save_path}")
    print("="*60)

    # wget 参数说明：
    # -c 断点续传 | --progress=bar:force 强制输出进度条
    # --show-progress 显示实时网速、剩余时间、文件大小
    # --tries 重试次数 | --timeout 超时时间
    cmd = [
        "wget",
        "-c",
        "--progress=bar:force",
        "--show-progress",
        "--tries", str(WGET_RETRY_COUNT),
        "--timeout", "30",
        url,
        "-O", save_path,
        "--header", f"Authorization: Bearer {HF_TOKEN}"
    ]
    # 流式实时打印wget输出，实时看到网速进度
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in iter(proc.stdout.readline, ""):
        print(line.rstrip())
    proc.wait()

    # 下载状态校验
    if proc.returncode == 0:
        if os.path.exists(save_path) and os.path.getsize(save_path) > 0:
            print(f"\n✅ 文件下载成功：{os.path.basename(save_path)} | 文件大小: {os.path.getsize(save_path)/1024/1024:.2f} MB")
        else:
            print(f"\n❌ 下载完成但文件为空，删除损坏文件准备重下")
            os.remove(save_path)
    else:
        print(f"\n❌ 下载失败，wget返回码: {proc.returncode}")

# 启动保活后台线程（守护线程，脚本结束自动销毁）
keep_alive_thread = threading.Thread(target=colab_keep_alive, daemon=True)
keep_alive_thread.start()
print("🔋 Colab内核保活线程已启动，全程防止闲置断开")
time.sleep(2)

# ---------------------- 下载任务队列 ----------------------
# 1. LTX 2.3 FP8 主模型
url1 = "https://huggingface.co/Lightricks/LTX-2.3-fp8/resolve/main/ltx-2.3-22b-dev-fp8.safetensors"
save1 = f"{COMFY_PATH}/models/checkpoints/ltx-2.3-22b-dev-fp8.safetensors"
download_file(url1, save1)

# 2. 空间超分模型
url2 = "https://huggingface.co/Lightricks/LTX-2.3/resolve/main/ltx-2.3-spatial-upscaler-x2-1.1.safetensors"
save2 = f"{COMFY_PATH}/models/latent_upscale_models/ltx-2.3-spatial-upscaler-x2-1.1.safetensors"
download_file(url2, save2)

# 3. LTX LoRA
url3 = "https://huggingface.co/Comfy-Org/ltx-2.3/resolve/main/split_files/loras/ltx_2.3_22b_distilled_1.1_lora_dynamic_fro09_avg_rank_111_bf16.safetensors"
save3 = f"{COMFY_PATH}/models/loras/ltx_2.3_22b_distilled_1.1_lora_dynamic_fro09_avg_rank_111_bf16.safetensors"
download_file(url3, save3)

# 4. Gemma LoRA
url4 = "https://huggingface.co/Comfy-Org/ltx-2/resolve/main/split_files/loras/gemma-3-12b-it-abliterated_lora_rank64_bf16.safetensors"
save4 = f"{COMFY_PATH}/models/loras/gemma-3-12b-it-abliterated_lora_rank64_bf16.safetensors"
download_file(url4, save4)

# ---------------------- 下载完成校验 ----------------------
print("\n\n🎉 全部下载任务执行完毕，文件清单校验：")
print("\n【Checkpoints目录】")
subprocess.run(["ls", "-lh", f"{COMFY_PATH}/models/checkpoints/"])
print("\n【latent_upscale_models目录】")
subprocess.run(["ls", "-lh", f"{COMFY_PATH}/models/latent_upscale_models/"])
print("\n【loras目录】")
subprocess.run(["ls", "-lh", f"{COMFY_PATH}/models/loras/"])

print("\n📌 保活线程将持续运行，如需关闭内核可手动停止单元格")
# 主线程循环挂起，持续保活，不会自动结束
while True:
    time.sleep(3600)

🔋 Colab内核保活线程已启动，全程防止闲置断开

[保活心跳] 2026-07-07 07:37:56 内核保活成功

【开始任务】https://huggingface.co/Lightricks/LTX-2.3-fp8/resolve/main/ltx-2.3-22b-dev-fp8.safetensors
【保存路径】/root/comfy/ComfyUI/models/checkpoints/ltx-2.3-22b-dev-fp8.safetensors
--2026-07-07 07:37:58--  https://huggingface.co/Lightricks/LTX-2.3-fp8/resolve/main/ltx-2.3-22b-dev-fp8.safetensors
Resolving huggingface.co (huggingface.co)... 18.164.174.118, 18.164.174.55, 18.164.174.17, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.118|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.aws.cdn.hf.co/xet-bridge-us/69aa086a8bf763d7e27cd33e/94d11e676b844dd40d74196f059f8b6fdc0fa5c6a07ed9db72d91be06910640a?X-Xet-Cas-Uid=public&user_id=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27ltx-2.3-22b-dev-fp8.safetensors%3B+filename%3D%22ltx-2.3-22b-dev-fp8.safetensors%22%3B&Expires=1783413478&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5hd3MuY2RuLmhmLmNvL3hld

In [ ]:
!pip install comfy-cli

In [ ]:
!yes | comfy install --nvidia

In [ ]:
!wget --content-disposition -O "/root/comfy/ComfyUI/models/checkpoints/noobaiXLNAIXL_vPred10Version.safetensors" "https://huggingface.co/misri/noobaiXLNAIXL_vPred10Version/resolve/main/noobaiXLNAIXL_vPred10Version.safetensors"

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

In [ ]:
!mkdir -p /root/comfy/ComfyUI/user/default/workflows
!curl -L -o /root/comfy/ComfyUI/user/default/workflows/示例.json https://raw.githubusercontent.com/4evergr8/ComfyColab/refs/heads/main/%E7%A4%BA%E4%BE%8B.json

In [ ]:
!nohup cloudflared tunnel --url http://127.0.0.1:8188 --protocol http2 > output.log 2>&1 &

In [ ]:
!rm -rf /root/comfy/ComfyUI/output
!ln -s /content/drive/MyDrive /root/comfy/ComfyUI/output

In [ ]:
!python3 /root/comfy/ComfyUI/main.py